# DeepLabCut 工具箱 - Docker
https://github.com/DeepLabCut/DeepLabCut

Nath\*、Mathis\* 等人撰写的文章：《*Using DeepLabCut for markerless pose estimation during behavior across species*》（使用 DeepLabCut 进行跨物种行为中的无标记姿态估计）

本教程演示了在您自己的项目中使用 DeepLabCut 所需的必要步骤。
这里展示了最简单的实现代码，但请注意，其中许多函数都具备额外的功能，因此请务必查阅 概述 和 协议论文！

本教程演示了如何使用 Docker 容器来完成以下操作：
- 训练一个网络
- 评估一个网络
- 分析一个新的视频

**注意：** 本教程的前提是您已经有一个包含已标记数据的项目文件夹！

## 让我们来看一下 Docker 环境的相关信息：

In [ ]:
!nvcc --version

In [ ]:
import torch

# Let's make sure we see a GPU:
print(torch.__version__)
print(torch.cuda.is_available())

## 在这里开始训练 DeepLabCut 以及分析新视频！

In [ ]:
# GUIs don't work on in Docker (or the cloud), so label your data locally on your computer! 
# This notebook is for you to train and run video analysis!
import os

os.environ["DLClight"] = "True"

In [ ]:
# now we are ready to train!
import deeplabcut

### 更改为你的路径：

In [ ]:
# change to yours!
path_config_file = '/home/mackenzie/DEEPLABCUT/DeepLabCut/examples/Reaching-Mackenzie-2018-08-30/config.yaml'

## 注意，如果您移动了项目，或者下载了此项目并正在使用演示代码，您需要编辑 `config.yaml` 文件中的项目路径！

前往项目文件夹，使用任何文本编辑器（例如 Ubuntu 中的 gedit）打开该 yaml 文件。

(description): project_path:  项目的完整路径（如果您需要将项目移动到集群/服务器/另一台计算机，或者移动到您计算机上的另一个目录，请编辑此项）

## 创建训练数据集

这个函数用于生成 DeepLabCut 所需的训练数据。用户可以在 `config.yaml` 文件中设置训练集大小的比例（该比例基于 hd5 文件中所有已标注图像的数量）。在创建数据集的过程中，用户可以创建多个不同的“洗牌”（shuffles，即数据划分方案）。

运行此脚本后，训练数据集将被创建，并保存在项目目录下的 **`training-datasets`** 子目录中。

此函数还会：
1. 在 **`dlc-models-pytorch`** 下创建新的子目录。
2. 创建一个 `pytorch_config.yaml` 文件，该文件定义了模型架构，并包含了用于训练网络的各种参数。对于绝大多数应用场景，我们推荐使用默认设置即可。如需了解更多关于可以设置的变量信息，请查阅 [官方文档](https://deeplabcut.github.io/DeepLabCut/docs/pytorch/pytorch_config.html)！

In [ ]:
deeplabcut.create_training_dataset(path_config_file, Shuffles=[1])

## 开始训练

此函数针对训练数据集的特定洗牌（shuffle）来训练网络。

In [ ]:
deeplabcut.train_network(
    path_config_file,
    shuffle=1,
    save_epochs=2,
    displayiters=5,
)

# This will run until you stop it (CTRL+C), or hit "STOP" icon, or when it
# hits the end (default, 200 epochs).

# If you end training before it hits the end, you will see what looks like
# an error message, but it's not an error - don't worry....


## 开始评估

此函数用于对训练好的模型进行评估，评估范围是针对**特定次数的洗牌（shuffle/shuffles）和特定状态（state）**，也可以在数据集（images）上的**所有状态**上进行评估，并将评估结果作为 `.csv` 文件存储在 `evaluation-results-pytorch` 目录下的一个子目录中。

In [ ]:
deeplabcut.evaluate_network(path_config_file)

# Here you want to see a low pixel error! Of course, it can only
# be as good as the labeler, so be sure your labels are good!

## 存在一个可选的优化步骤

- 如果您的像素错误率不够低，请查阅协议指南，了解如何优化您的网络！
- 您需要在 **DOCKER 外部** 调整标签！我们建议您返回来训练和分析视频...
- 请参阅仓库和协议说明，了解如何优化您的数据！

```markdown
## 开始分析视频

此函数用于分析新视频。用户可以选择评估结果中最佳的模型，并为 `config.yaml` 文件中的变量 **snapshotindex** 指定正确的快照（snapshot）索引。如果未进行指定，则默认使用最近的一个快照来分析视频。

分析结果将存储在与视频位于同一目录下的 hd5 文件中。
```

In [ ]:
videofile_path = [
    "/home/mackenzie/DEEPLABCUT/DeepLabCut/examples/Reaching-Mackenzie-2018-08-30/videos/MovieS2_Perturbation_noLaser_compressed.avi"
]  # Enter the list of videos to analyze.
deeplabcut.analyze_videos(path_config_file, videofile_path)

## 创建带标签的视频

此函数用于可视化目的，可用于创建包含网络预测标签的 `.mp4` 格式视频。此视频将保存到与原始视频相同的目录下。

In [ ]:
deeplabcut.create_labeled_video(path_config_file, videofile_path)

## 绘制分析视频的轨迹

此函数会绘制整个视频中所有身体部位的轨迹。每个身体部位都由一个唯一的颜色来标识。

In [ ]:
%matplotlib notebook 
# for making interactive plots.
# deeplabcut.plot_trajectories(path_config_file, videofile_path, plotting=True)
deeplabcut.plot_trajectories(path_config_file, videofile_path, showfigures=True)